In [17]:
import xarray as xr
import numpy as np
import functions
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib as mpl
import seaborn as sns
import cmcrameri
from scipy import stats
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.markers import MarkerStyle
import statsmodels.api as sm
import pymannkendall as mk
from global_land_mask import globe

In [19]:
rpath = '/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/observations_reanalysis_data/'

## Processing and re-indexing of data

### Land surface temperature

In [15]:
filepath = rpath + 'temperature/ERA5-Land_t2m_Arctic.nc'
temp_ERA5_Land = xr.open_dataset(filepath)

# Re-index
temp_ERA5_Land = temp_ERA5_Land.rename({'latitude':'lat', 'longitude':'lon', 'valid_time':'time'})
temp_ERA5_Land = temp_ERA5_Land.reindex(lat=list(reversed(temp_ERA5_Land.lat)))
lons = np.array(temp_ERA5_Land.coords['lon'])
lons[np.where(lons<0)] = 360 + lons[np.where(lons<0)]
temp_ERA5_Land.coords['lon'] = lons
temp_ERA5_Land = temp_ERA5_Land.sortby(temp_ERA5_Land.lon)

temp_ERA5_Land.to_netcdf(rpath+'temperature/ERA5-Land_t2m_Arctic_reindexed.nc')

### Soil moisture ERA5-Land

In [ ]:
# OPEN DATASET

filepath_1 = rpath + 'soil_moisture/SM_ERA5_1970_2025_2levels.nc'
SM_ERA5_1 = xr.open_dataset(filepath_1)

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 168, lat: 301, lon: 3600)
Coordinates:
  * time       (time) datetime64[ns] 1kB 1970-06-01 1970-07-01 ... 2025-08-01
    expver     (time) <U4 3kB ...
  * lat        (lat) float64 2kB 60.0 60.1 60.2 60.3 ... 89.7 89.8 89.9 90.0
  * lon        (lon) float64 29kB 0.1 0.2 0.3 0.4 ... 359.7 359.8 359.9 360.0
    number     int64 8B ...
Data variables:
    swvl1      (time, lat, lon) float32 728MB ...
    swvl2      (time, lat, lon) float32 728MB ...
    swvl_20cm  (time, lat, lon) float32 728MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-30T14:21 GRIB to CDM+CF via cfgrib-0.9.1...

In [ ]:
# OPEN DATASET

filepath_1 = rpath + 'soil_moisture/SM_level1_ERA5-Land.nc'
filepath_2 = rpath + 'soil_moisture/SM_level2_ERA5-Land.nc'
SM_ERA5_1 = xr.open_dataset(filepath_1)
SM_ERA5_2 = xr.open_dataset(filepath_2)

SM_ERA5_1 = SM_ERA5_1.sel(valid_time=slice('1970-01-01','2025-09-01'))
SM_ERA5_1 = SM_ERA5_1.sel(valid_time=SM_ERA5_1.valid_time.dt.season=='JJA')
SM_ERA5 = xr.merge([SM_ERA5_1, SM_ERA5_2],compat='override')

In [6]:
# CREATE SOIL LEVEL FOR 10 CM

# Soil level 1: 0-7 cm
# Soil level 2: 7-28 cm
weighting_fraction = (10-7)/(28-7)
SM_ERA5['swvl_10cm'] = SM_ERA5['swvl1']+weighting_fraction*SM_ERA5['swvl2']

In [5]:
# CREATE SOIL LEVEL FOR 20 CM

# Soil level 1: 0-7 cm
# Soil level 2: 7-28 cm
weighting_fraction = (20-7)/(28-7)
SM_ERA5['swvl_20cm'] = SM_ERA5['swvl1']+((20-7)/(28-7))*SM_ERA5['swvl2']

In [7]:
# RE-INDEX

SM_ERA5 = SM_ERA5.rename({'latitude':'lat', 'longitude':'lon', 'valid_time':'time'})
SM_ERA5_1970_2025 = SM_ERA5.sel(time=slice('1970-01-01','2025-09-01'))
SM_ERA5_1970_2025 = SM_ERA5_1970_2025.reindex(lat=list(reversed(SM_ERA5_1970_2025.lat)))
lons = np.array(SM_ERA5_1970_2025.coords['lon'])
lons[np.where(lons<0)] = 360 + lons[np.where(lons<0)]
SM_ERA5_1970_2025.coords['lon'] = lons
SM_ERA5_1970_2025 = SM_ERA5_1970_2025.sortby(SM_ERA5_1970_2025.lon)
SM_ERA5_1970_2025.to_netcdf(rpath+'SM_ERA5_1970_2025_10cm_20cm.nc')

### Soil moisture Wang & Mao (2021)

In [57]:
filepath='/nird/datalake/NS9560K/diagnostics/ILAMB-Data/DATA/mrsos/WangMao/mrsos_olc.nc'

SM_WM = xr.open_dataset(filepath)
SM_WM_1970_2016 = SM_WM.sel(time=slice('1970-01-01','2016-12-31'),lat=slice(59,90))
SM_WM_1970_2016_masked = SM_WM_1970_2016.where(SM_WM_1970_2016['mrsos'] < 1000)
SM_WM_1970_2016_masked = SM_WM_1970_2016_masked.drop_vars('time_bounds')
lons = np.array(SM_WM_1970_2016_masked.coords['lon'])
lons[np.where(lons<0)] = 360 + lons[np.where(lons<0)]
SM_WM_1970_2016_masked.coords['lon'] = lons
SM_WM_1970_2016_masked = SM_WM_1970_2016_masked.sortby(SM_WM_1970_2016_masked.lon)
SM_WM_1970_2016_masked.to_netcdf(rpath+'SM_WangMao_1970_2016_masked.nc')